# DTI
Используем пока только Open Targets версии 26.09

In [1]:
%pip install -q duckdb==1.5.6 pandas==3.1.1 pyarrow==23.0.1 rdkit==2026.9.1 "fsspec[http]==2026.9.0"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path

import duckdb
import fsspec
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors

## Скачиваем данные

In [3]:
RELEASE = "26.09"
BASE_URL = f"https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/{RELEASE}/output"

DATA_DIR_RELPATH = "../data/"
RAW_DIR = Path(f"{DATA_DIR_RELPATH}/OpenTargets") / RELEASE
OUTPUT_DIR = Path(f"{DATA_DIR_RELPATH}/dataset")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USED_DATASETS = ["drug_molecule", "target", "drug_mechanism_of_action"]

fs = fsspec.filesystem("https")
for dataset in USED_DATASETS:
    folder = RAW_DIR / dataset
    folder.mkdir(parents=True, exist_ok=True)
    files = fs.glob(f"{BASE_URL}/{dataset}/*.parquet")
    if not files:
        raise FileNotFoundError(f"Нет файлов датасета: {dataset}")
    for url in files:
        fs.get_file(url, str(folder / url.rsplit("/", 1)[-1]))

In [4]:
con = duckdb.connect()
for dataset in USED_DATASETS:
    con.read_parquet(str(RAW_DIR / dataset / "*.parquet")).create_view(dataset)
    print(dataset)
    display(con.sql(f"DESCRIBE {dataset}").df())

drug_molecule


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,canonicalSmiles,VARCHAR,YES,None,None,None
2,inchiKey,VARCHAR,YES,None,None,None
3,molblock,VARCHAR,YES,None,None,None
4,drugType,VARCHAR,YES,None,None,None
5,name,VARCHAR,YES,None,None,None
6,parentId,VARCHAR,YES,None,None,None
7,synonyms,"STRUCT(""label"" VARCHAR, ""source"" VARCHAR)[]",YES,None,None,None
8,tradeNames,"STRUCT(""label"" VARCHAR, ""source"" VARCHAR)[]",YES,None,None,None
9,crossReferences,"STRUCT(""source"" VARCHAR, ids VARCHAR[])[]",YES,None,None,None


target


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,approvedSymbol,VARCHAR,YES,None,None,None
2,biotype,VARCHAR,YES,None,None,None
3,genomicLocation,"STRUCT(chromosome VARCHAR, ""start"" BIGINT, ""en...",YES,None,None,None
4,approvedName,VARCHAR,YES,None,None,None
5,go,"STRUCT(id VARCHAR, ""source"" VARCHAR, evidence ...",YES,None,None,None
6,hallmarks,"STRUCT(attributes STRUCT(pmid BIGINT, descript...",YES,None,None,None
7,synonyms,"STRUCT(""label"" VARCHAR, ""source"" VARCHAR)[]",YES,None,None,None
8,symbolSynonyms,"STRUCT(""label"" VARCHAR, ""source"" VARCHAR)[]",YES,None,None,None
9,nameSynonyms,"STRUCT(""label"" VARCHAR, ""source"" VARCHAR)[]",YES,None,None,None


drug_mechanism_of_action


,column_name,column_type,null,key,default,extra
0,mechanismOfAction,VARCHAR,YES,None,None,None
1,actionType,VARCHAR,YES,None,None,None
2,chemblIds,VARCHAR[],YES,None,None,None
3,references,"STRUCT(""source"" VARCHAR, ids VARCHAR[], urls V...",YES,None,None,None
4,targetName,VARCHAR,YES,None,None,None
5,targetType,VARCHAR,YES,None,None,None
6,targets,VARCHAR[],YES,None,None,None


## Вспомогантельные функции

In [5]:
def save_duckdb_parquet(table_name: str):
    file_path = str(OUTPUT_DIR / f"{table_name}.parquet")
    con.table(table_name).write_parquet(file_path)
    print(f"Файл сохранен: {file_path}")

## Формируем таблицу compounds

In [6]:
parents_chain = dict(
    con.sql("select id, parentId from drug_molecule").fetchall()
)

In [7]:
small_molecules_df = (
    con.sql(
    """
    select id, name, canonicalSmiles, drugType, parentId
    from drug_molecule
    where drugType = 'Small molecule'
    """
    )
).df().set_index("id", drop=False)
small_molecules_df.index.name = None

In [8]:
small_molecules_df["molecule"] = small_molecules_df["canonicalSmiles"].map(
    lambda smiles: Chem.MolFromSmiles(smiles) 
    if isinstance(smiles, str) and smiles.strip() 
    else None
)
small_molecules_df = small_molecules_df.dropna(subset=["molecule"])

[17:57:19] WARNING: not removing hydrogen atom without neighbors
[17:57:19] WARNING: not removing hydrogen atom without neighbors
[17:57:19] WARNING: not removing hydrogen atom without neighbors
[17:57:19] WARNING: not removing hydrogen atom without neighbors
[17:57:19] WARNING: not removing hydrogen atom without neighbors


In [9]:
def get_parent_id(child_id):
    visited = set()
    curr_id = child_id
    while curr_id in parents_chain:
        if curr_id in visited:
            return None
        visited.add(curr_id)
        parent = parents_chain[curr_id]
        if parent is None or parent == "" or parent == curr_id:
            return curr_id
        curr_id = parent
    return None

child_to_parent = small_molecules_df["id"].map(get_parent_id)

In [10]:
compounds = small_molecules_df.loc[
    small_molecules_df.index.isin(child_to_parent.dropna().unique())
]
compounds = compounds.loc[
    compounds["molecule"].map(lambda molecule: len(Chem.GetMolFrags(molecule)) == 1)
].copy()

In [11]:
def get_molecular_features(molecule):
    return {
        "molecular_weight": Descriptors.MolWt(molecule),
        "logp": Crippen.MolLogP(molecule),
        "tpsa": rdMolDescriptors.CalcTPSA(molecule),
        "h_bond_donors": Lipinski.NumHDonors(molecule),
        "h_bond_acceptors": Lipinski.NumHAcceptors(molecule),
        "rotatable_bonds": Lipinski.NumRotatableBonds(molecule),
        "aromatic_rings": rdMolDescriptors.CalcNumAromaticRings(molecule),
    }
    

molecular_features = pd.DataFrame(
    compounds["molecule"].map(get_molecular_features).tolist(), index=compounds.index
)
compounds = pd.concat([compounds, molecular_features], axis=1)
compounds = compounds.drop(columns="molecule").sort_values("id").reset_index(drop=True)

In [12]:
compounds = compounds[
    [
        "id", "name", "canonicalSmiles", "drugType", "parentId",
        "molecular_weight", "logp", "tpsa",
        "h_bond_donors", "h_bond_acceptors",
        "rotatable_bonds", "aromatic_rings",
    ]
].sort_values("id").reset_index(drop=True)

con.register("compounds", compounds)

In [13]:
save_duckdb_parquet("compounds")

Файл сохранен: ../data/dataset/compounds.parquet


## Формируем таблицу proteins

In [14]:
con.sql("""
    create temp view enzymes as
    select id, approvedSymbol, approvedName, biotype,
           targetClass, subcellularLocations, go, pathways,
           list_transform(
               list_filter(targetClass, c -> c.level = 'l1' and c.label = 'Enzyme'),
               c -> c.id
           ) as enzyme_paths
    from target
    where biotype = 'protein_coding' and len(enzyme_paths) > 0;
""")

In [15]:
con.sql("""
    create temp view enzyme_families as
    with classified as (
        select * exclude (enzyme_paths),
            list_filter(
                targetClass,
                c -> list_contains(enzyme_paths, c.id)
            ) as enzyme_classes
        from enzymes
    )
    select * exclude (enzyme_classes),
        list_sort(list_distinct(list_transform(
            list_filter(enzyme_classes, c -> c.level = 'l2'),
            c -> c.label
        ))) as enzyme_subfamilies,
        list_sort(list_distinct(list_transform(
            list_filter(enzyme_classes, c -> c.level = 'l3'),
            c -> c.label
        ))) as enzyme_l3
    from classified;
""")

In [16]:
con.sql("""
    create temp table proteins as
    select *,
        coalesce(
            list_unique(list_transform(list_filter(go, g -> g.aspect = 'F'), g -> g.id))
            , 0
        ) as n_go_functions,
        coalesce(
            list_unique(list_transform(list_filter(go, g -> g.aspect = 'P'), g -> g.id))
            , 0
        ) as n_go_processes,
        coalesce(
            list_unique(list_transform(pathways, p -> p.pathwayId))
            , 0
        ) as n_pathways
    from enzyme_families
    order by id;
""")

In [17]:
save_duckdb_parquet("proteins")

Файл сохранен: ../data/dataset/proteins.parquet


## Формируем таблицу compound_protein

In [18]:
con.sql("""
    create temp view single_protein_links as
    select unnest(chemblIds) as original_compound_id, 
           targets[1] as target_id,
           actionType, mechanismOfAction, targetType
    from drug_mechanism_of_action
    where targetType = 'single protein' and len(targets) = 1;
""")

In [19]:
con.register("compound_mapping", 
    child_to_parent.rename("compound_id").rename_axis("original_compound_id").reset_index()
)

con.sql("""
    create temp view selected_links as
    select m.compound_id, l.*
    from single_protein_links as l
    join compound_mapping as m using (original_compound_id)
    join compounds as c on c.id = m.compound_id
    join proteins as p on p.id = l.target_id;
""")

In [20]:
con.sql("""
    create temp table compound_protein as
    select compound_id, target_id,
           list_sort(list_distinct(list(actionType))) as actionType,
           list_sort(list_distinct(list(mechanismOfAction))) as mechanismOfAction,
           first(targetType) as targetType
    from selected_links
    group by compound_id, target_id
    order by compound_id, target_id;
""")

In [21]:
save_duckdb_parquet("compound_protein")

Файл сохранен: ../data/dataset/compound_protein.parquet
